In [54]:
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.metrics import root_mean_squared_error



In [5]:
data=pd.read_csv("bigmart.csv")
data.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


In [6]:
data.info()
data.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 8523 entries, 0 to 8522
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Item_Identifier            8523 non-null   str    
 1   Item_Weight                7060 non-null   float64
 2   Item_Fat_Content           8523 non-null   str    
 3   Item_Visibility            8523 non-null   float64
 4   Item_Type                  8523 non-null   str    
 5   Item_MRP                   8523 non-null   float64
 6   Outlet_Identifier          8523 non-null   str    
 7   Outlet_Establishment_Year  8523 non-null   int64  
 8   Outlet_Size                6113 non-null   str    
 9   Outlet_Location_Type       8523 non-null   str    
 10  Outlet_Type                8523 non-null   str    
 11  Item_Outlet_Sales          8523 non-null   float64
dtypes: float64(4), int64(1), str(7)
memory usage: 799.2 KB


Item_Identifier                 0
Item_Weight                  1463
Item_Fat_Content                0
Item_Visibility                 0
Item_Type                       0
Item_MRP                        0
Outlet_Identifier               0
Outlet_Establishment_Year       0
Outlet_Size                  2410
Outlet_Location_Type            0
Outlet_Type                     0
Item_Outlet_Sales               0
dtype: int64

In [7]:
data["Outlet_Size"].value_counts()

Outlet_Size
Medium    2793
Small     2388
High       932
Name: count, dtype: int64

In [8]:
data["Item_Weight"].value_counts()

Item_Weight
12.150    86
17.600    82
13.650    77
11.800    76
9.300     68
          ..
5.210      2
7.685      1
9.420      1
6.520      1
5.400      1
Name: count, Length: 415, dtype: int64

In [9]:
data['Item_Weight'] = data.groupby('Item_Identifier')['Item_Weight'].transform(
    lambda x: x.fillna(x.mean())
)

data["Outlet_Size"]=data.groupby("Outlet_Type")["Outlet_Size"].transform(
    lambda x: x.fillna(x.mode()[0])
)

In [10]:
data=data.dropna()

In [11]:
data.isna().sum()


Item_Identifier              0
Item_Weight                  0
Item_Fat_Content             0
Item_Visibility              0
Item_Type                    0
Item_MRP                     0
Outlet_Identifier            0
Outlet_Establishment_Year    0
Outlet_Size                  0
Outlet_Location_Type         0
Outlet_Type                  0
Item_Outlet_Sales            0
dtype: int64

In [12]:
data.drop(columns=["Item_Identifier","Outlet_Identifier"],inplace=True)




In [13]:
data.columns

Index(['Item_Weight', 'Item_Fat_Content', 'Item_Visibility', 'Item_Type',
       'Item_MRP', 'Outlet_Establishment_Year', 'Outlet_Size',
       'Outlet_Location_Type', 'Outlet_Type', 'Item_Outlet_Sales'],
      dtype='str')

In [14]:
data["Item_Fat_Content"].value_counts()
data["Outlet_Location_Type"].value_counts()

Outlet_Location_Type
Tier 3    3347
Tier 2    2785
Tier 1    2387
Name: count, dtype: int64

In [15]:
data["Item_Fat_Content"]=data["Item_Fat_Content"].replace({
    "LF":"Low Fat",
    "low fat":"Low Fat",
    "reg":"Regular"
})

In [16]:
data["Outlet_Size"].value_counts()
data["Outlet_Location_Type"].value_counts()
data["Outlet_Type"].value_counts()
data["Item_Type"].value_counts()

Item_Type
Fruits and Vegetables    1232
Snack Foods              1199
Household                 910
Frozen Foods              855
Dairy                     681
Canned                    649
Baking Goods              647
Health and Hygiene        520
Soft Drinks               445
Meat                      425
Breads                    251
Hard Drinks               214
Others                    169
Starchy Foods             148
Breakfast                 110
Seafood                    64
Name: count, dtype: int64

In [17]:
data["Item_Type"]=data["Item_Type"].replace({
   "Snack Foods":"Foods",
   "Frozen Foods":"Foods",
   "Meat":"Foods",
   "Seafood":"Foods",
   "Breakfast":"Foods",
   "Breads":"Foods",
   "Fruits and Vegetables":"Foods",
   "Dairy":"Drinks",
   "Soft Drinks":"Drinks",
   "Hard Drinks":"Drinks",
   "Baking Goods":"Foods",
   "Canned":"Foods",
   "Starchy Foods":"Foods"
})
data["Item_Type"].value_counts()

Item_Type
Foods                 5580
Drinks                1340
Household              910
Health and Hygiene     520
Others                 169
Name: count, dtype: int64

In [18]:
data_enc=pd.get_dummies(data)
data_enc.head()

,Item_Weight,Item_Visibility,Item_MRP,Outlet_Establishment_Year,Item_Outlet_Sales,Item_Fat_Content_Low Fat,Item_Fat_Content_Regular,Item_Type_Drinks,Item_Type_Foods,Item_Type_Health and Hygiene,...,Outlet_Size_High,Outlet_Size_Medium,Outlet_Size_Small,Outlet_Location_Type_Tier 1,Outlet_Location_Type_Tier 2,Outlet_Location_Type_Tier 3,Outlet_Type_Grocery Store,Outlet_Type_Supermarket Type1,Outlet_Type_Supermarket Type2,Outlet_Type_Supermarket Type3
0,9.30,0.016047,249.8092,1999,3735.1380,True,False,True,False,False,...,False,True,False,True,False,False,False,True,False,False
1,5.92,0.019278,48.2692,2009,443.4228,False,True,True,False,False,...,False,True,False,False,False,True,False,False,True,False
2,17.50,0.016760,141.6180,1999,2097.2700,True,False,False,True,False,...,False,True,False,True,False,False,False,True,False,False
3,19.20,0.000000,182.0950,1998,732.3800,False,True,False,True,False,...,False,False,True,False,False,True,True,False,False,False
4,8.93,0.000000,53.8614,1987,994.7052,True,False,False,False,False,...,True,False,False,False,False,True,False,True,False,False


In [19]:
data_enc.dtypes
for col in data_enc.columns:
    if data_enc[col].dtype=="bool":
        data_enc[col]=data_enc[col].astype(int)

In [20]:
data_enc.dtypes

Item_Weight                      float64
Item_Visibility                  float64
Item_MRP                         float64
Outlet_Establishment_Year          int64
Item_Outlet_Sales                float64
Item_Fat_Content_Low Fat           int64
Item_Fat_Content_Regular           int64
Item_Type_Drinks                   int64
Item_Type_Foods                    int64
Item_Type_Health and Hygiene       int64
Item_Type_Household                int64
Item_Type_Others                   int64
Outlet_Size_High                   int64
Outlet_Size_Medium                 int64
Outlet_Size_Small                  int64
Outlet_Location_Type_Tier 1        int64
Outlet_Location_Type_Tier 2        int64
Outlet_Location_Type_Tier 3        int64
Outlet_Type_Grocery Store          int64
Outlet_Type_Supermarket Type1      int64
Outlet_Type_Supermarket Type2      int64
Outlet_Type_Supermarket Type3      int64
dtype: object

In [21]:
data_enc.head()

,Item_Weight,Item_Visibility,Item_MRP,Outlet_Establishment_Year,Item_Outlet_Sales,Item_Fat_Content_Low Fat,Item_Fat_Content_Regular,Item_Type_Drinks,Item_Type_Foods,Item_Type_Health and Hygiene,...,Outlet_Size_High,Outlet_Size_Medium,Outlet_Size_Small,Outlet_Location_Type_Tier 1,Outlet_Location_Type_Tier 2,Outlet_Location_Type_Tier 3,Outlet_Type_Grocery Store,Outlet_Type_Supermarket Type1,Outlet_Type_Supermarket Type2,Outlet_Type_Supermarket Type3
0,9.30,0.016047,249.8092,1999,3735.1380,1,0,1,0,0,...,0,1,0,1,0,0,0,1,0,0
1,5.92,0.019278,48.2692,2009,443.4228,0,1,1,0,0,...,0,1,0,0,0,1,0,0,1,0
2,17.50,0.016760,141.6180,1999,2097.2700,1,0,0,1,0,...,0,1,0,1,0,0,0,1,0,0
3,19.20,0.000000,182.0950,1998,732.3800,0,1,0,1,0,...,0,0,1,0,0,1,1,0,0,0
4,8.93,0.000000,53.8614,1987,994.7052,1,0,0,0,0,...,1,0,0,0,0,1,0,1,0,0


In [23]:
data_enc.corr()

,Item_Weight,Item_Visibility,Item_MRP,Outlet_Establishment_Year,Item_Outlet_Sales,Item_Fat_Content_Low Fat,Item_Fat_Content_Regular,Item_Type_Drinks,Item_Type_Foods,Item_Type_Health and Hygiene,...,Outlet_Size_High,Outlet_Size_Medium,Outlet_Size_Small,Outlet_Location_Type_Tier 1,Outlet_Location_Type_Tier 2,Outlet_Location_Type_Tier 3,Outlet_Type_Grocery Store,Outlet_Type_Supermarket Type1,Outlet_Type_Supermarket Type2,Outlet_Type_Supermarket Type3
Item_Weight,1.000000,-0.009173,0.025975,-0.013426,0.013168,0.026798,-0.026798,-0.027983,-0.024783,0.009709,...,0.009862,0.004967,-0.010906,0.005709,-0.016020,0.010137,0.006931,-0.007679,-0.000156,0.004460
Item_Visibility,-0.009173,1.000000,-0.001155,-0.074325,-0.128297,-0.046902,0.046902,0.019629,0.037708,-0.053860,...,-0.041824,-0.081070,0.103026,0.060991,-0.068561,0.009768,0.285947,-0.143856,-0.034555,-0.051416
Item_MRP,0.025975,-0.001155,1.000000,0.004599,0.567803,-0.006358,0.006358,0.000429,-0.004211,-0.041721,...,0.002341,-0.004195,0.002496,-0.001319,0.001759,-0.000476,-0.004273,0.004507,0.003754,-0.006054
Outlet_Establishment_Year,-0.013426,-0.074325,0.004599,1.000000,-0.049083,-0.003753,0.003753,0.003669,0.001079,-0.003147,...,-0.453904,-0.015713,0.300517,-0.201894,0.540678,-0.333656,-0.281206,0.244323,0.466356,-0.537667
Item_Outlet_Sales,0.013168,-0.128297,0.567803,-0.049083,1.000000,-0.018974,0.018974,-0.010007,0.016565,-0.025578,...,0.024197,0.204442,-0.208664,-0.110985,0.058322,0.046038,-0.411549,0.108919,-0.038048,0.311089
Item_Fat_Content_Low Fat,0.026798,-0.046902,-0.006358,-0.003753,-0.018974,1.000000,-1.000000,0.093373,-0.362851,0.188126,...,0.001996,-0.004454,0.002958,-0.003046,0.003001,-0.000081,0.003255,-0.001071,-0.002267,0.000422
Item_Fat_Content_Regular,-0.026798,0.046902,0.006358,0.003753,0.018974,-1.000000,1.000000,-0.093373,0.362851,-0.188126,...,-0.001996,0.004454,-0.002958,0.003046,-0.003001,0.000081,-0.003255,0.001071,0.002267,-0.000422
Item_Type_Drinks,-0.027983,0.019629,0.000429,0.003669,-0.010007,0.093373,-0.093373,1.000000,-0.595303,-0.110155,...,0.005578,-0.006770,0.002895,0.005410,0.003390,-0.008231,-0.000188,0.012045,-0.005143,-0.013014
Item_Type_Foods,-0.024783,0.037708,-0.004211,0.001079,0.016565,-0.362851,0.362851,-0.595303,1.000000,-0.351319,...,-0.008280,0.011856,-0.006007,-0.001925,0.002004,-0.000155,-0.007206,-0.005178,0.004878,0.010707
Item_Type_Health and Hygiene,0.009709,-0.053860,-0.041721,-0.003147,-0.025578,0.188126,-0.188126,-0.110155,-0.351319,1.000000,...,0.006457,-0.000315,-0.003765,-0.008410,-0.004177,0.011745,0.001406,-0.005589,0.002132,0.004886


In [24]:
data_enc.head()

,Item_Weight,Item_Visibility,Item_MRP,Outlet_Establishment_Year,Item_Outlet_Sales,Item_Fat_Content_Low Fat,Item_Fat_Content_Regular,Item_Type_Drinks,Item_Type_Foods,Item_Type_Health and Hygiene,...,Outlet_Size_High,Outlet_Size_Medium,Outlet_Size_Small,Outlet_Location_Type_Tier 1,Outlet_Location_Type_Tier 2,Outlet_Location_Type_Tier 3,Outlet_Type_Grocery Store,Outlet_Type_Supermarket Type1,Outlet_Type_Supermarket Type2,Outlet_Type_Supermarket Type3
0,9.30,0.016047,249.8092,1999,3735.1380,1,0,1,0,0,...,0,1,0,1,0,0,0,1,0,0
1,5.92,0.019278,48.2692,2009,443.4228,0,1,1,0,0,...,0,1,0,0,0,1,0,0,1,0
2,17.50,0.016760,141.6180,1999,2097.2700,1,0,0,1,0,...,0,1,0,1,0,0,0,1,0,0
3,19.20,0.000000,182.0950,1998,732.3800,0,1,0,1,0,...,0,0,1,0,0,1,1,0,0,0
4,8.93,0.000000,53.8614,1987,994.7052,1,0,0,0,0,...,1,0,0,0,0,1,0,1,0,0


In [27]:
#Scaling input features
data_enc.columns

Index(['Item_Weight', 'Item_Visibility', 'Item_MRP',
       'Outlet_Establishment_Year', 'Item_Outlet_Sales',
       'Item_Fat_Content_Low Fat', 'Item_Fat_Content_Regular',
       'Item_Type_Drinks', 'Item_Type_Foods', 'Item_Type_Health and Hygiene',
       'Item_Type_Household', 'Item_Type_Others', 'Outlet_Size_High',
       'Outlet_Size_Medium', 'Outlet_Size_Small',
       'Outlet_Location_Type_Tier 1', 'Outlet_Location_Type_Tier 2',
       'Outlet_Location_Type_Tier 3', 'Outlet_Type_Grocery Store',
       'Outlet_Type_Supermarket Type1', 'Outlet_Type_Supermarket Type2',
       'Outlet_Type_Supermarket Type3'],
      dtype='str')

In [32]:
data_enc.describe()

,Item_Weight,Item_Visibility,Item_MRP,Outlet_Establishment_Year,Item_Outlet_Sales,Item_Fat_Content_Low Fat,Item_Fat_Content_Regular,Item_Type_Drinks,Item_Type_Foods,Item_Type_Health and Hygiene,...,Outlet_Size_High,Outlet_Size_Medium,Outlet_Size_Small,Outlet_Location_Type_Tier 1,Outlet_Location_Type_Tier 2,Outlet_Location_Type_Tier 3,Outlet_Type_Grocery Store,Outlet_Type_Supermarket Type1,Outlet_Type_Supermarket Type2,Outlet_Type_Supermarket Type3
count,8519.000000,8519.000000,8519.000000,8519.000000,8519.000000,8519.000000,8519.000000,8519.000000,8519.000000,8519.000000,...,8519.000000,8519.000000,8519.000000,8519.000000,8519.000000,8519.000000,8519.000000,8519.000000,8519.000000,8519.000000
mean,12.875420,0.066112,141.010019,1997.837892,2181.188779,0.647494,0.352506,0.157295,0.655006,0.061040,...,0.109403,0.327503,0.563094,0.280197,0.326916,0.392886,0.127010,0.654654,0.108933,0.109403
std,4.646098,0.051586,62.283594,8.369105,1706.511093,0.477779,0.477779,0.364100,0.475394,0.239418,...,0.312162,0.469330,0.496032,0.449122,0.469114,0.488421,0.333004,0.475509,0.311573,0.312162
min,4.555000,0.000000,31.290000,1985.000000,33.290000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,8.785000,0.026983,93.844900,1987.000000,834.247400,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,12.650000,0.053925,143.047000,1999.000000,1794.331000,1.000000,0.000000,0.000000,1.000000,0.000000,...,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000
75%,16.850000,0.094558,185.676600,2004.000000,3100.630600,1.000000,1.000000,0.000000,1.000000,0.000000,...,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000
max,21.350000,0.328391,266.888400,2009.000000,13086.964800,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [34]:
#feature engineering
data_enc['Outlet_Age'] = 2009 - data_enc['Outlet_Establishment_Year']
data_enc.drop("Outlet_Establishment_Year",axis=1,inplace=True)

In [35]:
data_enc.head()

,Item_Weight,Item_Visibility,Item_MRP,Item_Outlet_Sales,Item_Fat_Content_Low Fat,Item_Fat_Content_Regular,Item_Type_Drinks,Item_Type_Foods,Item_Type_Health and Hygiene,Item_Type_Household,...,Outlet_Size_Medium,Outlet_Size_Small,Outlet_Location_Type_Tier 1,Outlet_Location_Type_Tier 2,Outlet_Location_Type_Tier 3,Outlet_Type_Grocery Store,Outlet_Type_Supermarket Type1,Outlet_Type_Supermarket Type2,Outlet_Type_Supermarket Type3,Outlet_Age
0,9.30,0.016047,249.8092,3735.1380,1,0,1,0,0,0,...,1,0,1,0,0,0,1,0,0,10
1,5.92,0.019278,48.2692,443.4228,0,1,1,0,0,0,...,1,0,0,0,1,0,0,1,0,0
2,17.50,0.016760,141.6180,2097.2700,1,0,0,1,0,0,...,1,0,1,0,0,0,1,0,0,10
3,19.20,0.000000,182.0950,732.3800,0,1,0,1,0,0,...,0,1,0,0,1,1,0,0,0,11
4,8.93,0.000000,53.8614,994.7052,1,0,0,0,0,1,...,0,0,0,0,1,0,1,0,0,22


In [36]:
Y=data_enc["Item_Outlet_Sales"]
X=data_enc.drop("Item_Outlet_Sales",axis=1)

In [38]:

# First split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, Y, test_size=0.3, random_state=42
)
# Split temp into 50% validation, 50% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

In [39]:
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (5963, 21)
Validation: (1278, 21)
Test: (1278, 21)


In [40]:
features_to_scale=["Item_Weight","Item_Visibility","Item_MRP","Outlet_Age"]
scaler=StandardScaler()
scaler.fit(X_train[features_to_scale])
X_train[features_to_scale]=scaler.transform(X_train[features_to_scale])
X_val[features_to_scale]=scaler.transform(X_val[features_to_scale])
X_test[features_to_scale]=scaler.transform(X_test[features_to_scale])

Manual implementation of  Gradient Descent

In [51]:
def gradient_descent(X, y, lr=0.01, epochs=1000):
    n_samples, n_features = X.shape
    
    W = np.zeros(n_features)
    b = 0
    
    for epoch in range(epochs):
        
        # 1. Prediction
        y_pred = np.dot(X, W) + b
        
        # 2. Error
        error = y_pred - y
        
        # 3. Gradients
        dW = (2/n_samples) * np.dot(X.T, error)
        db = (2/n_samples) * np.sum(error)
        
        # 4. Update
        W -= lr * dW
        b -= lr * db
        
    return W, b

W, b = gradient_descent(X_train.values, y_train.values, lr=0.001, epochs=1000000)

In [49]:
def predict(X, W, b):
    return np.dot(X, W) + b

In [52]:
y_val_pred = predict(X_val.values, W, b)

mse_val = np.mean((y_val - y_val_pred)**2)
rmse_val = np.sqrt(mse_val)

print("Validation RMSE:", rmse_val)

Validation RMSE: 1122.7491425617732


Using Sklearn

In [ ]:
model=LinearRegression()
model.fit(X_train,y_train)
val_predict=model.predict(X_val)
rmse = root_mean_squared_error(y_val, val_predict)


In [57]:
print(rmse)

1122.6964040398823
